# Code Review Agent (Notebook)

Minimal agentic demo that reviews a pull request and posts a comment using:
- An **OpenAI-compliant** inference endpoint secured with API Key
- MCP Endpoint secured with API Key. **2 MCP tools**: `get_pull_request_files`, `create_issue_comment`

Set env vars (or override in the config cell): `OPENAI_API_BASE`, `OPENAI_API_KEY`, optionally `MCP_PROXY_URL`, `MCP_PROXY_BEARER_TOKEN`, `OPENAI_MODEL`.

## 1. Imports and config

In [ ]:
import json
import os

import requests
from openai import OpenAI

# Config (from env; override below for notebook)
MCP_URL = os.environ.get("MCP_PROXY_URL", "http://localhost:8080").rstrip("/")
MCP_BEARER_TOKEN = (os.environ.get("MCP_PROXY_BEARER_TOKEN", "") or "").strip()
OPENAI_BASE = os.environ.get("OPENAI_API_BASE", "")
OPENAI_KEY = os.environ.get("OPENAI_API_KEY", "dummy")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o")

TOOL_NAMES = ["get_pull_request_files", "create_issue_comment"]

# Optional: override for notebook (uncomment and set)
# MCP_URL = "http://localhost:8080"
# MCP_BEARER_TOKEN = ""
# OPENAI_BASE = "https://your-endpoint/v1"
# OPENAI_KEY = "your-api-key"
# OPENAI_MODEL = "gpt-4o"

print(f"MCP_URL: {MCP_URL}")
print(f"OPENAI_BASE: {OPENAI_BASE or '(not set)'}")
print(f"OPENAI_MODEL: {OPENAI_MODEL}")

## 2. MCP client

In [ ]:
def _mcp_headers():
    headers = {"Content-Type": "application/json"}
    if MCP_BEARER_TOKEN:
        headers["Authorization"] = f"Bearer {MCP_BEARER_TOKEN}"
    return headers


def mcp_request(method: str, params: dict | None = None) -> dict:
    payload = {"jsonrpc": "2.0", "id": 1, "method": method}
    if params is not None:
        payload["params"] = params
    resp = requests.post(f"{MCP_URL}/", json=payload, headers=_mcp_headers(), timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if "error" in data:
        raise RuntimeError(data["error"].get("message", data["error"]))
    return data.get("result", {})


def mcp_initialize():
    return mcp_request("initialize", {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "code-review-agent", "version": "1.0"},
    })


def mcp_list_tools() -> list:
    result = mcp_request("tools/list")
    all_tools = result.get("tools", [])
    return [t for t in all_tools if t.get("name") in TOOL_NAMES]


def mcp_call_tool(name: str, arguments: dict) -> str:
    result = mcp_request("tools/call", {"name": name, "arguments": arguments})
    for part in result.get("content", []):
        if part.get("type") == "text":
            return part.get("text", "")
    return ""

## 3. MCP → OpenAI tools and system prompt

In [ ]:
def mcp_tools_to_openai(mcp_tools: list) -> list:
    openai_tools = []
    for t in mcp_tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t.get("description", ""),
                "parameters": t.get("inputSchema", {"type": "object", "properties": {}}),
            },
        })
    return openai_tools


SYSTEM_PROMPT = """You are a code review agent. You have access to two tools:

1. get_pull_request_files(owner, repo, pull_number) - Get the list of files changed in a pull request with their diffs and change statistics. Use this first to see what code changed.

2. create_issue_comment(owner, repo, issue_number, body) - Create a comment on the pull request. Use this to post your review. For a PR, issue_number is the same as the pull request number.

Your task: Review the code for the given PR and then post a single, concise review comment summarizing your findings (what looks good, any suggestions or concerns). Use the tools in order: first get the PR files, then write the review comment. After posting the comment, reply with a short confirmation to the user."""

## 4. Agent loop

In [ ]:
def run_agent(owner: str, repo: str, pull_number: int) -> str:
    if not OPENAI_BASE:
        raise ValueError("Set OPENAI_API_BASE to your OpenAI-compliant inference endpoint URL")

    client = OpenAI(base_url=OPENAI_BASE, api_key=OPENAI_KEY)
    mcp_initialize()
    mcp_tool_list = mcp_list_tools()
    openai_tools = mcp_tools_to_openai(mcp_tool_list)

    user_message = (
        f"Please review the code for this pull request and add a comment with your review.\n"
        f"Repository: {owner}/{repo}\n"
        f"Pull request number: {pull_number}\n"
        f"Use get_pull_request_files first, then create_issue_comment to post your review."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    final_content = ""
    max_turns = 10

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            tools=openai_tools,
            tool_choice="auto",
        )
        choice = response.choices[0]
        msg = choice.message
        finish = choice.finish_reason

        if msg.content:
            final_content = (msg.content or "").strip()

        if finish == "stop" and not getattr(msg, "tool_calls", None):
            return final_content or "Review flow finished."

        tool_calls = getattr(msg, "tool_calls", None) or []
        if not tool_calls:
            return final_content or "Review flow finished."

        assistant_msg = {
            "role": "assistant",
            "content": msg.content or None,
            "tool_calls": [
                {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in tool_calls
            ],
        }
        messages.append(assistant_msg)

        for tc in tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments) if isinstance(tc.function.arguments, str) else tc.function.arguments
            except json.JSONDecodeError:
                args = {}
            try:
                result = mcp_call_tool(name, args)
            except Exception as e:
                result = json.dumps({"error": str(e)})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return final_content or "Max turns reached."

## 5. Run the agent

In [ ]:
owner = "octocat"      # repo owner
repo = "hello-world"   # repo name
pull_number = 123       # PR number

result = run_agent(owner, repo, pull_number)
print(result)